# 4. Modeling: Perbandingan Model Klasifikasi Kerapian Alas Kaki

Lima baseline machine learning dibandingkan dengan **StratifiedGroupKFold**. Setiap fold memisahkan seluruh foto dari kelompok yang sama dan StandardScaler dipelajari ulang hanya pada bagian latih fold. Model dipilih berdasarkan rata-rata macro-F1, kemudian dilatih ulang pada semua kelompok pengembangan; Kelompok 2 tetap disisihkan untuk evaluasi akhir.

In [ ]:
import sys
import shutil
from pathlib import Path
import json
import joblib
import pandas as pd
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, make_scorer
from sklearn.model_selection import StratifiedGroupKFold, cross_validate
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler


def find_project_dir():
    for base in (Path.cwd(), *Path.cwd().parents):
        for candidate in (base, base / Path("src/1-project/proyek_sain_data/src/10-project_citra")):
            if (candidate / "3-data_preparation").is_dir() and (candidate / "6-deployment").is_dir():
                return candidate.resolve()
    raise FileNotFoundError("Folder proyek 10-project_citra tidak ditemukan dari direktori kerja.")


PROJECT_DIR = find_project_dir()
DATA_PROCESSED = PROJECT_DIR / "3-data_preparation" / "data" / "processed"
MODEL_DIR = PROJECT_DIR / "4-modeling" / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
manifest = pd.read_csv(DATA_PROCESSED / "dataset_manifest.csv")
training_df = manifest.loc[manifest["split"] != "test"].reset_index(drop=True)
feature_cols = [
    column for column in training_df.columns
    if column not in {"filename", "rel_path", "kelompok", "class", "label", "split", "width", "height", "aspect_ratio"}
]
X = training_df[feature_cols]
y = training_df["label"].to_numpy()
groups = training_df["kelompok"].to_numpy()
cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
folds = list(cv.split(X, y, groups))
print(f"Data pengembangan: {len(training_df)} citra dari {training_df['kelompok'].nunique()} kelompok; fold group-aware: {len(folds)}")

---
## 4.1 Pelatihan & Perbandingan Model pada Validation Set


In [ ]:
models = {
    "ExtraTrees": ExtraTreesClassifier(n_estimators=300, max_depth=8, class_weight="balanced", random_state=42),
    "RandomForest": RandomForestClassifier(n_estimators=300, max_depth=8, class_weight="balanced", random_state=42),
    "GradientBoosting": GradientBoostingClassifier(n_estimators=100, max_depth=2, learning_rate=0.05, random_state=42),
    "SVC": CalibratedClassifierCV(
        estimator=SVC(kernel="rbf", C=1.0, class_weight="balanced", random_state=42),
        method="sigmoid", cv=3,
    ),
    "LogisticRegression": LogisticRegression(max_iter=2000, class_weight="balanced", random_state=42),
}

results = []
best_f1 = -1.0
best_model_name = None
best_model = None
for name, candidate in models.items():
    cv_results = cross_validate(
        make_pipeline(StandardScaler(), candidate), X, y, cv=folds,
        scoring={
            "macro_f1": make_scorer(f1_score, average="macro", zero_division=0),
            "accuracy": "accuracy",
            "precision_macro": make_scorer(precision_score, average="macro", zero_division=0),
            "recall_macro": make_scorer(recall_score, average="macro", zero_division=0),
        },
        n_jobs=1,
    )
    macro_f1 = float(cv_results["test_macro_f1"].mean())
    results.append({
        "Model": name,
        "CV Accuracy Mean": float(cv_results["test_accuracy"].mean()),
        "CV Accuracy Std": float(cv_results["test_accuracy"].std()),
        "CV Macro F1 Mean": macro_f1,
        "CV Macro F1 Std": float(cv_results["test_macro_f1"].std()),
        "CV Precision Macro": float(cv_results["test_precision_macro"].mean()),
        "CV Recall Macro": float(cv_results["test_recall_macro"].mean()),
    })
    if macro_f1 > best_f1:
        best_f1 = macro_f1
        best_model_name = name
        best_model = candidate

df_res = pd.DataFrame(results).sort_values("CV Macro F1 Mean", ascending=False)
display(df_res)
print(f"Model terpilih dari group CV: {best_model_name} (macro-F1={best_f1:.3f})")

---
## 4.2 Simpan Bobot Model Terbaik


In [ ]:
final_scaler = StandardScaler().fit(X.to_numpy())
X_scaled = final_scaler.transform(X.to_numpy())
best_model.fit(X_scaled, y)

joblib.dump(best_model, MODEL_DIR / "best_model.joblib")
joblib.dump(best_model, MODEL_DIR / "best_model.pkl")
joblib.dump(final_scaler, MODEL_DIR / "scaler.joblib")
with (MODEL_DIR / "feature_columns.json").open("w", encoding="utf-8") as file:
    json.dump(feature_cols, file, indent=2)
df_res.to_json(MODEL_DIR / "model_comparison.json", orient="records", indent=2)

deploy_dir = PROJECT_DIR / "6-deployment"
deploy_dir.mkdir(parents=True, exist_ok=True)
shutil.copy2(PROJECT_DIR / "feature_extraction.py", deploy_dir / "feature_extraction.py")
joblib.dump(best_model, deploy_dir / "best_model.joblib")
joblib.dump(final_scaler, deploy_dir / "scaler.joblib")
with (deploy_dir / "feature_columns.json").open("w", encoding="utf-8") as file:
    json.dump(feature_cols, file, indent=2)
print(f"Model {best_model_name} dilatih ulang pada semua {len(training_df)} citra dari kelompok pengembangan.")